# Etapa 1:

## Qual o problema socioeconômico que você está investigando?

O Nordeste apresenta um clima bem heterogêneo junto de uma biodiversidade ambiental vasta, tendo uma diferença grande nos regimes de precipitação, temperatura, umidade e disponibilidade hídrica. Essa variação pode afetar justamente as queimadas através de ressecamento da vegetação e do material de combustível, provocando maior intensidade nos focos de queimadas. Nesse contexto, o problema socioeconômico investigado está relacionado aos impactos que esses eventos podem provocar sobre a população e sobre as atividades econômicas da região, especialmente na agricultura, pecuária, saúde pública e conservação dos recursos naturais. Queimadas de maior intensidade podem causar perdas de áreas produtivas, degradação ambiental, aumento da emissão de poluentes atmosféricos e maior demanda por ações de combate e prevenção por parte do poder público.

## Por que ele é relevante para o Nordeste brasileiro?

Por causa da presença de extensas áreas sujeitas a períodos de estiagem e elevada variabilidade das condições meteorológicas, além da importância das atividades agropecuárias para diversos municípios da região. A combinação entre baixa precipitação, temperaturas elevadas, baixa umidade e disponibilidade de material combustível pode favorecer condições de maior risco de fogo. Dessa forma, a análise conjunta dos dados meteorológicos disponibilizados pelo Instituto Nacional de Meteorologia (INMET) e dos dados de focos de calor e Potência Radiativa do Fogo (FRP) do Instituto Nacional de Pesquisas Espaciais (INPE), entre 2020 e 2024, permite identificar padrões espaciais e temporais, períodos críticos e áreas com maior frequência ou intensidade de queimadas.

## Que tipo de decisão um gestor público poderia tomar com base nos resultados do seu modelo?

A identificação de períodos e regiões com maior probabilidade de ocorrência ou intensidade de queimadas poderia orientar a distribuição de equipes de combate a incêndios, a intensificação da fiscalização, a definição de áreas prioritárias para monitoramento e a emissão de alertas preventivos. Além disso, os resultados poderiam auxiliar no planejamento de políticas ambientais, agrícolas e de proteção civil, permitindo que os recursos públicos sejam direcionados de forma mais eficiente para os locais e períodos de maior risco.

## Hipótese inicial:

A precipitação acumulada nos dias anteriores (7, 15 ou 30 dias) à ocorrência do foco apresenta associação negativa mais forte com o FRP do que a precipitação registrada no próprio dia, de modo que períodos antecedentes mais secos estão associados a focos de maior intensidade.

A precipitação acumulada nos 7, 15 ou 30 dias anteriores à ocorrência do foco apresentará associação negativa estatisticamente significativa com o FRP (p < 0,05), sendo sua correlação, em valor absoluto, superior à observada entre a precipitação do próprio dia e o FRP.

A hipótese será rejeitada caso nenhuma das janelas de precipitação antecedente apresente associação negativa significativa com o FRP ou caso sua associação não seja superior à da precipitação registrada no próprio dia.

## Hipóteses secundárias:

### Hipótese 1:

Áreas caracterizadas por condições recorrentes de baixa precipitação e baixa umidade apresentam agrupamentos espaciais de focos com FRP elevado, indicando que a intensidade das queimadas não se distribui aleatoriamente no território nordestino.

Áreas classificadas no quartil inferior de precipitação acumulada e umidade relativa apresentarão maior proporção de focos classificados no quartil superior de FRP e autocorrelação espacial positiva e estatisticamente significativa, medida pelo índice de Moran global (I > 0; p < 0,05) e Moran Local/LISA.

A hipótese será rejeitada caso não seja detectada autocorrelação espacial positiva significativa ou caso as áreas de menor precipitação e umidade não apresentem maior concentração de focos com FRP elevado.

### Hipótese 2:

Velocidades mais elevadas do vento estão associadas a maiores valores de FRP, e essa associação é intensificada quando ocorrem simultaneamente baixa umidade relativa e baixa precipitação acumulada nos dias anteriores.

A velocidade do vento apresentará associação positiva e estatisticamente significativa com o FRP (p < 0,05), e o efeito estimado do vento será maior nas observações simultaneamente classificadas no quartil inferior de umidade relativa e precipitação antecedente do que nas demais condições.

A hipótese será rejeitada caso a velocidade do vento não apresente associação positiva significativa com o FRP ou caso o termo de interação entre vento e condições secas não indique aumento significativo do FRP.

# Etapa 2

In [4]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('INPE-MLlib')
    .master('local[*]')
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

Vamos ler primeiramente os dados de ambos as fontes (inpe e inmet) e analisar os primeiros registros

In [5]:
df_inpe = (
    spark.read
    .option("header", True)
    .option('sep', ',')
    .option('inferSchema', True)
    .csv("../data/bronze/inpe/*.csv")
)

In [6]:
df_inpe.show()

+--------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
|            DataHora| Satelite|  Pais|        Estado|         Municipio|         Bioma|DiaSemChuva|Precipitacao|RiscoFogo| FRP|           Latitude|          Longitude|
+--------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
| 2020/01/01 00:45:28|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.3|      0.2|NULL|-25.187599182128906| -49.39350128173828|
|﻿2020/01/01 00:45:28|  METOP-B|Brasil|        PARANÁ| RIO BRANCO DO SUL|Mata Atlântica|       11.0|         3.2|      0.2|NULL|-25.185699462890625|-49.383201599121094|
| 2020/01/01 00:45:53|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.4|      0.2|NULL|-25.225500106811523| -49.38

In [7]:
df_inpe.printSchema()

root
 |-- DataHora: string (nullable = true)
 |-- Satelite: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Bioma: string (nullable = true)
 |-- DiaSemChuva: double (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- RiscoFogo: double (nullable = true)
 |-- FRP: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



Percebe-se que a coluna de DataHora veio como string, vamos tratar isso e transformar em timestamp:

DataHora: timestamp
DiaSemChuva: integer
Precipitacao: double
RiscoFogo: double
FRP (Fire Radiative Power): double
Latitude: double
Longitude: double

In [8]:
from pyspark.sql import functions as F

df_inpe = df_inpe.withColumn(
    "DataHora",
    F.coalesce(
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy/MM/dd HH:mm:ss")
        ),
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
)

In [9]:
df_inpe.printSchema()

root
 |-- DataHora: timestamp (nullable = true)
 |-- Satelite: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Bioma: string (nullable = true)
 |-- DiaSemChuva: double (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- RiscoFogo: double (nullable = true)
 |-- FRP: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



Com as tipagens corretas vamos analisar agora as estatísticas descritivas mais comuns

In [10]:
df_inpe.describe().show()

+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|summary| Satelite|    Pais|   Estado|      Municipio|   Bioma|       DiaSemChuva|      Precipitacao|         RiscoFogo|              FRP|           Latitude|         Longitude|
+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|  count| 26976571|26976571| 26976571|       26976571|26976507|          26478762|          26478762|          26478762|         21335115|           26976571|          26976571|
|   mean|     NULL|    NULL|     NULL|           NULL|    NULL|19.166153047487644| 0.751714091844774|-8.485602798191671|33.89365710941816|-10.182877124779395| -52.6529186162472|
| stddev|     NULL|    NULL|     NULL|           NULL|    NULL|101.65792629735161|3.6642243221536015| 95.71982

Observa-se que o inpe usa -999 para valores nulos, o que acaba prejudicando a média e o desvio-padrão das colunas que tem isso (dia sem chvua e risco fogo), vamos trocar esses valores por NULL para deixá-los mais corretos

In [11]:
df_inpe = (
    df_inpe.replace(
        -999,
        None,
        subset=['DiaSemChuva','RiscoFogo']
    )
)

In [13]:
df_inpe.describe().show()

+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|summary| Satelite|    Pais|   Estado|      Municipio|   Bioma|       DiaSemChuva|      Precipitacao|         RiscoFogo|              FRP|           Latitude|         Longitude|
+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|  count| 26976571|26976571| 26976571|       26976571|26976507|          26252452|          26478762|          26233778|         21335115|           26976571|          26976571|
|   mean|     NULL|    NULL|     NULL|           NULL|    NULL|27.943282974100857| 0.751714091844774|0.7643107706410313|33.89365710941816|-10.182877124779395| -52.6529186162472|
| stddev|     NULL|    NULL|     NULL|           NULL|    NULL| 37.54738917154817|3.6642243221536015|0.3355730

In [14]:
df_inpe.count()

26976571

Agora com os dados aparentemente corretos vamos ver os do inmet para fazer o join espaço-temporal

O detalhe aqui dos dados do inmet é o seguinte, temos um exemplo de estrutura:

Nome: JEREMOABO
Codigo Estacao: A450
Latitude: -10.08083332
Longitude: -38.34583333
Altitude: 261
Situacao: Pane
Data Inicial: 2020-01-01
Data Final: 2024-12-31
Periodicidade da Medicao: Horaria

Data Medicao;Hora Medicao;PRECIPITACAO TOTAL, HORARIO(mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA(mB);PRESSAO ATMOSFERICA REDUZIDA NIVEL DO MAR, AUT(mB);PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT)(mB);PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT)(mB);RADIACAO GLOBAL(Kj/m²);TEMPERATURA DA CPU DA ESTACAO(°C);TEMPERATURA DO AR - BULBO SECO, HORARIA(°C);TEMPERATURA DO PONTO DE ORVALHO(°C);TEMPERATURA MAXIMA NA HORA ANT. (AUT)(°C);TEMPERATURA MINIMA NA HORA ANT. (AUT)(°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT)(°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT)(°C);TENSAO DA BATERIA DA ESTACAO(V);UMIDADE REL. MAX. NA HORA ANT. (AUT)(%);UMIDADE REL. MIN. NA HORA ANT. (AUT)(%);UMIDADE RELATIVA DO AR, HORARIA(%);VENTO, DIRECAO HORARIA (gr)(° (gr));VENTO, RAJADA MAXIMA(m/s);VENTO, VELOCIDADE HORARIA(m/s);
2020-01-01;0000;0;983,2;1013,3;983,2;982,2;-1,9;26;24,3;21,1;25,1;24,2;22,2;20,9;12,6;84;81;83;36;4,4;1,7;
2020-01-01;0100;0;983,4;1013,6;983,4;983,2;-3,5;26;23,3;20,8;24,3;23,3;21,2;20,7;12,6;86;81;86;119;4,3;2;
2020-01-01;0200;0;983;1013,2;983,5;983;-3,5;25;23;21,1;23,3;23;21,1;20,8;12,6;89;86;89;122;3,1;1,1;
2020-01-01;0300;0;982,3;1012,5;983;982,3;-3,2;25;23,1;20,3;23,1;23;21,1;20,2;12,6;89;84;84;140;2,2;,6;
2020-01-01;0400;0;981,3;1011,5;982,3;981,3;-3;25;22,6;20,1;23,1;22,6;20,3;19,7;12,5;86;82;86;334;2,1;1,2;
2020-01-01;0500;0;981,3;1011,5;981,4;981,2;-3,3;24;22,9;19,6;22,9;22,6;20,3;19,5;12,5;87;81;82;334;2,2;,7;

Para uma estrutura assim, vamos querer pegar pelo menos as informações iniciais alí e colocar no dataframe também, então vamos fazer duas leituras de dataframes, e colocar o código que está no nome do arquivo como coluna, para depois fazer o join e deixar as informações todas em um dataframe só

A primeira leitura dos dados:

In [110]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("data_medicao", StringType(), True),
    StructField("hora_medicao", StringType(), True),
    StructField("precipitacao_total_horario_mm", DoubleType(), True),
    StructField("pressao_atmosferica_estacao_horaria_mb", DoubleType(), True),
    StructField("pressao_atmosferica_nivel_mar_mb", DoubleType(), True),
    StructField("pressao_atmosferica_max_hora_ant_mb", DoubleType(), True),
    StructField("pressao_atmosferica_min_hora_ant_mb", DoubleType(), True),
    StructField("radiacao_global_kj_m2", DoubleType(), True),
    StructField("temperatura_cpu_estacao_c", DoubleType(), True),
    StructField("temperatura_ar_c", DoubleType(), True),
    StructField("temperatura_ponto_orvalho_c", DoubleType(), True),
    StructField("temperatura_max_hora_ant_c", DoubleType(), True),
    StructField("temperatura_min_hora_ant_c", DoubleType(), True),
    StructField("temperatura_orvalho_max_hora_ant_c", DoubleType(), True),
    StructField("temperatura_orvalho_min_hora_ant_c", DoubleType(), True),
    StructField("tensao_bateria_estacao_v", DoubleType(), True),
    StructField("umidade_relativa_max_hora_ant_pct", DoubleType(), True),
    StructField("umidade_relativa_min_hora_ant_pct", DoubleType(), True),
    StructField("umidade_relativa_ar_pct", DoubleType(), True),
    StructField("vento_direcao_horaria_graus", DoubleType(), True),
    StructField("vento_rajada_max_ms", DoubleType(), True),
    StructField("vento_velocidade_horaria_ms", DoubleType(), True),
])


# e aqui fazemos a leitura dos dados, já colocando também o nome do arquivo e trocando o null por valores nuloes mesmo, filtrando apenas as colunas que começam com data (colunas de dados com data)

df_inmet = (
    spark.read
    .option("header", False)
    .option("sep", ";")
    .option("nullValue", "null")
    .option("locale", "pt-BR")
    .schema(schema)
    .csv("../data/bronze/inmet/*.csv")
    .withColumn("arquivo", F.input_file_name())
    .filter(
        F.col("data_medicao").rlike(r"^\d{4}-\d{2}-\d{2}$")
    )
)

In [103]:
df_inmet.show(truncate=False)

+------------+------------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+-------------------------------------------------------------------------------------------------+
|data_medicao|hora_medicao|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalh

Agora ler um dataframe com as informações das estações:

In [111]:
from pyspark.sql import functions as F

df_estacoes = (
    spark.read
    .format("binaryFile")
    .load("../data/bronze/inmet/*.csv")

    .withColumn("arquivo", F.input_file_name())

    .withColumn(
        "texto",
        F.decode("content", "ISO-8859-1")
    )

    .select(
        "arquivo",

        F.regexp_extract(
            "texto",
            r"Nome:\s*([^\r\n]+)",
            1
        ).alias("nome"),

        F.regexp_extract(
            "texto",
            r"Codigo Estacao:\s*([^\r\n]+)",
            1
        ).alias("codigo_estacao"),

        F.regexp_extract(
            "texto",
            r"Latitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("latitude"),

        F.regexp_extract(
            "texto",
            r"Longitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("longitude"),

        F.regexp_extract(
            "texto",
            r"Altitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("altitude"),

        F.regexp_extract(
            "texto",
            r"Situacao:\s*([^\r\n]+)",
            1
        ).alias("situacao")
    )
)

In [105]:
df_estacoes.show(truncate=False)

+-------------------------------------------------------------------------------------------------+----------------+--------------+------------+------------+--------+--------+
|arquivo                                                                                          |nome            |codigo_estacao|latitude    |longitude   |altitude|situacao|
+-------------------------------------------------------------------------------------------------+----------------+--------------+------------+------------+--------+--------+
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A217_H_2020-01-01_2024-12-31.csv|FAROL de SANTANA|A217          |-2.27083332 |-43.62416666|9.87    |Pane    |
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A422_H_2020-01-01_2024-12-31.csv|ABROLHOS        |A422          |-17.96305555|-38.70333333|20.93   |Pane    |
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A344_H_2020-01-01_2024-12-31.csv|CALCANHAR       |A344

Agora com isso vamos juntar os dois dataframes pela coluna "arquivo" que é nossa chave temporária

In [112]:
df_inmet = (
    df_inmet
    .join(
        df_estacoes,
        on="arquivo",
        how="left"
    )
    .drop("arquivo")
)

In [113]:
df_inmet.show()

+------------+------------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+----------------+--------------+-----------+------------+--------+--------+
|data_medicao|hora_medicao|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalho_c|temperatura_max_hor

E com isso temos nosso dataframe final para o inmet na etapa bronze, vamos verificar as principais estatisticas:

In [114]:
df_inmet.printSchema()

root
 |-- data_medicao: string (nullable = true)
 |-- hora_medicao: string (nullable = true)
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- pressao_atmosferica_estacao_horaria_mb: double (nullable = true)
 |-- pressao_atmosferica_nivel_mar_mb: double (nullable = true)
 |-- pressao_atmosferica_max_hora_ant_mb: double (nullable = true)
 |-- pressao_atmosferica_min_hora_ant_mb: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- temperatura_cpu_estacao_c: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- temperatura_ponto_orvalho_c: double (nullable = true)
 |-- temperatura_max_hora_ant_c: double (nullable = true)
 |-- temperatura_min_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_max_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_min_hora_ant_c: double (nullable = true)
 |-- tensao_bateria_estacao_v: double (nullable = true)
 |-- umidade_relativa_max_hora_ant_pct: double (nullable 

In [123]:
df_inmet.describe().show()

+-------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+------------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+--------+--------------+------------------+-------------------+------------------+----------+
|summary|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|  temperatura_ar_c|temperatura_ponto_orvalho_c|temperatura_max_hora_ant_c|temper

Apenas a coluna de data que não está correta, vamos juntar "data" e "hora" em uma coluna DataHora para ficar igual ao outro dataframe do inpe em formato de timestamp

In [118]:
from pyspark.sql import functions as F

df_inmet = (
    df_inmet
    .withColumn(
        "DataHora",
        F.to_timestamp(
            F.concat(
                F.col("data_medicao"),
                F.col("hora_medicao")
            ),
            "yyyy-MM-ddHHmm"
        )
    )
    .drop('data_medicao', 'hora_medicao')
)

In [121]:
df_inmet.show()

+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+----------------+--------------+-----------+------------+--------+--------+-------------------+
|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalho_c|temperatura_max_hora_ant_c|temperatura_min_hora_ant

In [122]:
df_inmet.count()

6281160

In [124]:
df_inmet.printSchema()

root
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- pressao_atmosferica_estacao_horaria_mb: double (nullable = true)
 |-- pressao_atmosferica_nivel_mar_mb: double (nullable = true)
 |-- pressao_atmosferica_max_hora_ant_mb: double (nullable = true)
 |-- pressao_atmosferica_min_hora_ant_mb: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- temperatura_cpu_estacao_c: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- temperatura_ponto_orvalho_c: double (nullable = true)
 |-- temperatura_max_hora_ant_c: double (nullable = true)
 |-- temperatura_min_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_max_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_min_hora_ant_c: double (nullable = true)
 |-- tensao_bateria_estacao_v: double (nullable = true)
 |-- umidade_relativa_max_hora_ant_pct: double (nullable = true)
 |-- umidade_relativa_min_hora_ant_pct: double (nullable = true)
 |-- umidade_re

Agora vamos fazer o join temporal entre df_inpe e df_inmet por focos de queimada e estação meteorológica mais próxima
(por latitude/longitude e data), vamos usar a formula de Haversine para isso

baixar os dados:
inpe -> baixe de https://data.inpe.br/queimadas/bdqueimadas/#exportar-dados, selecione apenas estados do nordeste e periodo 2020 a 2024, baixe um zip por ano, depois extraia os csvs e coloque na pasta data/bronze/

inmet -> baixe de https://bdmep.inmet.gov.br/, siga o passo a passo, selecione virgula, dados horários, estações automáticas, selecione região